# Statistical Arbitrage — Research Notebook

This notebook provides an interactive exploration of the pairs-trading pipeline,
allowing step-by-step inspection of each stage.

**Pipeline stages covered:**
1. Data loading and exploration
2. Factor neutralisation diagnostics
3. Pair selection analysis
4. OU process fitting and threshold derivation
5. Kalman filter visualisation
6. Regime detection
7. Signal generation
8. Backtest results
9. Walk-forward validation

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from config import Config
from src import (
    data_loader,
    factor_neutralisation,
    pair_selection,
    ou_process,
    kalman_filter,
    regime_detector,
    strategy,
    backtest,
    analytics,
    walk_forward,
    visualisation,
)

import plotly.io as pio
pio.renderers.default = 'notebook'

print('Imports OK')

## 1. Configuration

In [ ]:
# Use a shorter date range for faster notebook execution
config = Config(
    start_date='2020-01-01',
    end_date='2024-12-31',
    top_n_pairs=5,
    ou_fitting_method='mle',
    use_ou_thresholds=True,
)

print(f'Universe: {len(config.tickers)} tickers across {len(set(config.sector_map.values()))} sectors')
print(f'Period: {config.start_date} → {config.end_date}')
print(f'Entry Z: {config.entry_z}, Exit Z: {config.exit_z}, Stop Z: {config.stop_z}')

## 2. Data Loading

In [ ]:
all_tickers = config.tickers + [config.market_ticker]
prices = data_loader.load(
    tickers=all_tickers,
    start=config.start_date,
    end=config.end_date,
    cache_dir='../.cache',
)

print(f'Price matrix: {prices.shape[0]} days × {prices.shape[1]} tickers')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
prices.tail(3)

In [ ]:
# Load VIX
try:
    vix_prices = data_loader.load(['%5EVIX'], start=config.start_date,
                                   end=config.end_date, cache_dir='../.cache')
    vix_series = vix_prices['^VIX'].dropna()
except Exception:
    vix_series = pd.Series(20.0, index=prices.index, name='^VIX')
print(f'VIX: mean={vix_series.mean():.1f}, max={vix_series.max():.1f}')

In [ ]:
market_prices = prices[config.market_ticker]
stock_prices  = prices.drop(columns=[config.market_ticker], errors='ignore')
all_returns   = data_loader.log_returns(prices)
market_returns = all_returns[config.market_ticker]
stock_returns  = all_returns.drop(columns=[config.market_ticker], errors='ignore')

# Normalised price chart for a few tickers
sample = ['XOM', 'CVX', 'JPM', 'GS', 'MSFT', 'GOOGL']
fig, ax = plt.subplots(figsize=(14, 5))
for t in sample:
    if t in stock_prices.columns:
        norm = stock_prices[t] / stock_prices[t].iloc[0] * 100
        ax.plot(norm.index, norm.values, label=t, lw=1.2)
ax.set_title('Normalised Prices (rebased to 100)', fontsize=12)
ax.legend(ncol=3)
ax.set_ylabel('Price index')
plt.tight_layout()
plt.show()

## 3. Factor Neutralisation

In [ ]:
neutral_prices, neutral_returns, betas_df = factor_neutralisation.neutralise(
    prices=stock_prices,
    returns=stock_returns,
    market_returns=market_returns,
    sector_map=config.sector_map,
    neutralise_market=config.neutralise_market,
    neutralise_sector=config.neutralise_sector,
)

print(f'Factor-neutral price matrix: {neutral_prices.shape}')
print(f'\nFactor betas summary:')
print(betas_df[['beta_mkt', 'beta_sec', 'r_squared']].describe().round(4))

In [ ]:
# Plot beta_mkt distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
betas_df['beta_mkt'].hist(bins=15, ax=ax1, color='steelblue', edgecolor='white')
ax1.set_title('Distribution of Market Beta (β_mkt)')
ax1.set_xlabel('β_mkt')
ax1.axvline(betas_df['beta_mkt'].mean(), color='red', lw=1.5, ls='--',
            label=f'mean={betas_df["beta_mkt"].mean():.3f}')
ax1.legend()

betas_df['beta_sec'].hist(bins=15, ax=ax2, color='steelblue', edgecolor='white')
ax2.set_title('Distribution of Sector Beta (β_sec)')
ax2.set_xlabel('β_sec')
plt.tight_layout()
plt.show()

## 4. Pair Selection

In [ ]:
selected_pairs = pair_selection.select_pairs(
    neutral_prices=neutral_prices,
    neutral_returns=neutral_returns,
    sector_map=config.sector_map,
    min_correlation=config.min_correlation,
    coint_pvalue=config.coint_pvalue,
    min_half_life=config.min_half_life,
    max_half_life=config.max_half_life,
    rolling_coint_window=config.rolling_coint_window,
    max_pairs_per_sector=config.max_pairs_per_sector,
)

print(f'Selected {len(selected_pairs)} pairs')
if not selected_pairs.empty:
    display(selected_pairs[[
        'ticker_y', 'ticker_x', 'sector_y', 'correlation',
        'eg_pvalue', 'half_life', 'score'
    ]].head(10).round(4))

In [ ]:
if not selected_pairs.empty:
    # Half-life distribution
    plt.figure(figsize=(8, 4))
    selected_pairs['half_life'].hist(bins=15, color='steelblue', edgecolor='white')
    plt.title('Distribution of OU Half-Lives (selected pairs)')
    plt.xlabel('Half-life (trading days)')
    plt.axvline(selected_pairs['half_life'].mean(), color='red', lw=1.5, ls='--',
                label=f'mean={selected_pairs["half_life"].mean():.1f} days')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 5. Kalman Filter & OU Process — Single Pair Diagnostics

In [ ]:
if not selected_pairs.empty:
    row = selected_pairs.iloc[0]
    TY, TX = row['ticker_y'], row['ticker_x']
    print(f'Analysing pair: {TY} / {TX}')
    print(f'Sector: {row["sector_y"]} | EG p-value: {row["eg_pvalue"]:.4f} | Half-life: {row["half_life"]:.1f} d')

    hedge, intercept, spread = kalman_filter.fit(
        y=neutral_prices[TY],
        x=neutral_prices[TX],
        delta=config.kalman_delta,
        vt=config.kalman_vt,
    )
    z = kalman_filter.zscore(spread, window=20)

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    # Row 1: Normalised prices
    norm_y = neutral_prices[TY] / neutral_prices[TY].iloc[0] * 100
    norm_x = neutral_prices[TX] / neutral_prices[TX].iloc[0] * 100
    axes[0].plot(norm_y.index, norm_y.values, label=TY, lw=1.2)
    axes[0].plot(norm_x.index, norm_x.values, label=TX, lw=1.2, ls='--')
    axes[0].set_title('Normalised Neutral Prices')
    axes[0].legend()

    # Row 2: Kalman hedge ratio
    axes[1].plot(hedge.index, hedge.values, label=f'β_t (Kalman hedge)', color='orange', lw=1.2)
    axes[1].axhline(row['hedge_ratio'], color='red', lw=1, ls='--',
                    label=f'OLS static β={row["hedge_ratio"]:.3f}')
    axes[1].set_title('Time-Varying Kalman Hedge Ratio β_t')
    axes[1].legend()

    # Row 3: Z-score
    axes[2].plot(z.index, z.values, label='Z-score', color='steelblue', lw=0.9)
    for level, ls, colour in [
        (config.entry_z, '--', 'green'), (-config.entry_z, '--', 'green'),
        (config.stop_z, ':', 'red'), (-config.stop_z, ':', 'red'),
    ]:
        axes[2].axhline(level, ls=ls, color=colour, lw=0.8)
    axes[2].set_title('Z-Score of Kalman Spread')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
if not selected_pairs.empty:
    try:
        ou_params = ou_process.fit(spread.dropna(), method=config.ou_fitting_method)
        print(ou_params)

        entry_thresh, exit_thresh = ou_process.optimal_thresholds(ou_params, config.sharpe_target)
        print(f'\nOU-optimal entry threshold: {entry_thresh:.4f} (spread units)')
        print(f'OU-optimal exit  threshold: {exit_thresh:.4f} (spread units)')

        fig_ou = ou_process.plot_diagnostics(spread.dropna(), ou_params)
        plt.show()
    except Exception as e:
        print(f'OU fitting failed: {e}')

## 6. Regime Detection

In [ ]:
regime_det = regime_detector.fit(
    market_returns=market_returns,
    vix_series=vix_series,
    vix_threshold=config.vix_threshold,
    hmm_n_states=config.hmm_n_states,
    use_hmm=config.use_hmm,
)
full_regime = regime_det.predict(neutral_prices.index)

risk_off_pct = (full_regime == 0).mean() * 100
print(f'Risk-off days: {risk_off_pct:.1f}% of total')
print(f'Normal  days: {100-risk_off_pct:.1f}% of total')

# Plot regime alongside VIX
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
vix_aligned = vix_series.reindex(full_regime.index, method='ffill')
ax1.plot(vix_aligned.index, vix_aligned.values, color='steelblue', lw=0.9)
ax1.axhline(config.vix_threshold, color='red', lw=1.2, ls='--',
            label=f'VIX threshold={config.vix_threshold}')
ax1.set_title('VIX Index')
ax1.legend()

ax2.fill_between(full_regime.index, full_regime.values, step='post',
                 alpha=0.7, color='green', label='Normal (1)')
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Risk-Off', 'Normal'])
ax2.set_title('Combined Regime')
ax2.legend()
plt.tight_layout()
plt.show()

## 7. Full Backtest — Top Pair

In [ ]:
if not selected_pairs.empty:
    row = selected_pairs.iloc[0]
    TY, TX = row['ticker_y'], row['ticker_x']

    # Signals
    ou_params_bt = None
    try:
        ou_params_bt = ou_process.fit(spread.dropna(), method=config.ou_fitting_method)
    except Exception:
        pass

    signals = strategy.generate_signals(
        zscore=z,
        regime=full_regime,
        entry_z=config.entry_z,
        exit_z=config.exit_z,
        stop_z=config.stop_z,
        ou_params=ou_params_bt,
        use_ou_thresholds=config.use_ou_thresholds,
        sharpe_target=config.sharpe_target,
        spread=spread,
    )

    ret_y = neutral_returns[TY]
    ret_x = neutral_returns[TX]

    pr = backtest.run_pair_backtest(
        ticker_y=TY, ticker_x=TX,
        returns_y=ret_y, returns_x=ret_x,
        signals=signals, regime=full_regime,
        hedge_ratio=hedge, spread=spread, zscore=z,
        transaction_cost=config.transaction_cost,
        ou_params=ou_params_bt,
    )

    pair_ts = analytics.pair_tearsheet(pr)
    print(f'=== {TY}/{TX} Pair Tearsheet ===')
    for k, v in pair_ts.items():
        print(f'  {k:<30} {v}')

    # Equity curve
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(pr.equity_curve.index, pr.equity_curve.values, color='steelblue', lw=1.5)
    ax.axhline(1.0, color='grey', lw=0.5, ls='--')
    ax.set_title(f'{TY}/{TX} Pair Equity Curve (net of costs)')
    ax.set_ylabel('Equity (rebased to 1.0)')
    plt.tight_layout()
    plt.show()

## 8. Portfolio Backtest

In [ ]:
from main import run_pipeline
import os
os.chdir('..')

portfolio = run_pipeline(config)

spy_equity = market_prices / market_prices.iloc[0]
ts = analytics.tearsheet(
    portfolio_equity=portfolio.equity_curve,
    spy_equity=spy_equity.reindex(portfolio.equity_curve.index, method='ffill'),
    pair_results=portfolio.pair_results,
    initial_capital=config.initial_capital,
)

print('\n=== Portfolio Tearsheet ===')
for k, v in ts.items():
    if not isinstance(v, dict):
        print(f'  {k:<32} {v}')

In [ ]:
spy_eq = spy_equity.reindex(portfolio.equity_curve.index, method='ffill')
fig = visualisation.equity_curve(portfolio.equity_curve, spy_eq)
fig.show()

In [ ]:
fig_dd = visualisation.drawdown_chart(portfolio.equity_curve)
fig_dd.show()

In [ ]:
monthly = analytics.monthly_returns_table(portfolio.equity_curve)
if not monthly.empty:
    fig_m = visualisation.monthly_returns_heatmap(monthly)
    fig_m.show()

## 9. Market Neutrality Check

The portfolio beta to SPY should be near zero — this is the fundamental
property of a pairs trading strategy.

In [ ]:
print(f'Portfolio beta to SPY:        {portfolio.portfolio_beta:.4f}')
print(f'Portfolio correlation to SPY: {portfolio.portfolio_correlation_to_spy:.4f}')

if abs(portfolio.portfolio_beta) < 0.10:
    print('✓ Near-zero beta confirms market neutrality')
else:
    print('⚠ Beta is not near-zero — check factor neutralisation')